In [1]:

! pip install --upgrade --quiet langchain langchain-community langchain-experimental langchain-groq langchain-neo4j neo4j

## connect to neo4j


In [2]:
from langchain_neo4j import Neo4jGraph

NEO4J_URI = "neo4j+s://5864e104.databases.neo4j.io"  # Replace with your actual Neo4j connection string
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "***REMOVED_NEO4J_PASSWORD***"
NEO4J_DATABASE = "neo4j"

graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE
)

print("Connected to Neo4j")

Connected to Neo4j


In [5]:
from langchain_groq import ChatGroq
import os
from getpass import getpass

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

In [6]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=1024,
    reasoning_effort="low",
    model_kwargs={"parallel_tool_calls":False}
)

In [7]:
from langchain_core.documents import Document

movie_texts = [
    "Christopher Nolan directed Inception. Leonardo DiCaprio acted in Inception. Inception is a Science Fiction movie released in 2010.",
    "Christopher Nolan directed Interstellar. Matthew McConaughey acted in Interstellar. Interstellar is a Science Fiction movie released in 2014.",
    "Christopher Nolan directed The Dark Knight. Christian Bale acted in The Dark Knight. The Dark Knight is an Action movie released in 2008."
]

documents = [Document(page_content=text) for text in movie_texts]

documents

[Document(metadata={}, page_content='Christopher Nolan directed Inception. Leonardo DiCaprio acted in Inception. Inception is a Science Fiction movie released in 2010.'),
 Document(metadata={}, page_content='Christopher Nolan directed Interstellar. Matthew McConaughey acted in Interstellar. Interstellar is a Science Fiction movie released in 2014.'),
 Document(metadata={}, page_content='Christopher Nolan directed The Dark Knight. Christian Bale acted in The Dark Knight. The Dark Knight is an Action movie released in 2008.')]

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)

chunks = text_splitter.split_documents(documents)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:", chunk.page_content)

c:\Users\windows\Desktop\projects\advanced rag\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chunk 1: Christopher Nolan directed Inception. Leonardo DiCaprio acted in Inception. Inception is a Science Fiction movie released in 2010.
Chunk 2: Christopher Nolan directed Interstellar. Matthew McConaughey acted in Interstellar. Interstellar is a Science Fiction movie released in 2014.
Chunk 3: Christopher Nolan directed The Dark Knight. Christian Bale acted in The Dark Knight. The Dark Knight is an Action movie released in 2008.


In [9]:
from langchain_experimental.graph_transformers import LLMGraphTransformer

llm_transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=["Person", "Movie", "Genre"],
    allowed_relationships=["DIRECTED", "ACTED_IN", "IN_GENRE"]
)

graph_documents = llm_transformer.convert_to_graph_documents(chunks)

C:\Users\windows\AppData\Local\Temp\ipykernel_15520\1861339708.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer


In [10]:

graph_documents

[GraphDocument(nodes=[Node(id='Christopher Nolan', type='Person', properties={}), Node(id='Leonardo Dicaprio', type='Person', properties={}), Node(id='Inception', type='Movie', properties={}), Node(id='Science Fiction', type='Genre', properties={})], relationships=[Relationship(source=Node(id='Christopher Nolan', type='Person', properties={}), target=Node(id='Inception', type='Movie', properties={}), type='DIRECTED', properties={}), Relationship(source=Node(id='Leonardo Dicaprio', type='Person', properties={}), target=Node(id='Inception', type='Movie', properties={}), type='ACTED_IN', properties={}), Relationship(source=Node(id='Inception', type='Movie', properties={}), target=Node(id='Science Fiction', type='Genre', properties={}), type='IN_GENRE', properties={})], source=Document(metadata={}, page_content='Christopher Nolan directed Inception. Leonardo DiCaprio acted in Inception. Inception is a Science Fiction movie released in 2010.')),
 GraphDocument(nodes=[Node(id='Christopher No

In [11]:
# Inspect extracted graph data
for i, graph_doc in enumerate(graph_documents):
    print(f"\n--- Graph Document {i+1} ---")
    print("Nodes:")
    for node in graph_doc.nodes:
        print(node)
    print("Relationships:")
    for rel in graph_doc.relationships:
        print(rel)



--- Graph Document 1 ---
Nodes:
id='Christopher Nolan' type='Person' properties={}
id='Leonardo Dicaprio' type='Person' properties={}
id='Inception' type='Movie' properties={}
id='Science Fiction' type='Genre' properties={}
Relationships:
source=Node(id='Christopher Nolan', type='Person', properties={}) target=Node(id='Inception', type='Movie', properties={}) type='DIRECTED' properties={}
source=Node(id='Leonardo Dicaprio', type='Person', properties={}) target=Node(id='Inception', type='Movie', properties={}) type='ACTED_IN' properties={}
source=Node(id='Inception', type='Movie', properties={}) target=Node(id='Science Fiction', type='Genre', properties={}) type='IN_GENRE' properties={}

--- Graph Document 2 ---
Nodes:
id='Christopher Nolan' type='Person' properties={}
id='Matthew Mcconaughey' type='Person' properties={}
id='Interstellar' type='Movie' properties={}
id='Science Fiction' type='Genre' properties={}
Relationships:
source=Node(id='Christopher Nolan', type='Person', properti

In [13]:
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True
)

print("Graph data added to Neo4j")


Graph data added to Neo4j


In [14]:
graph.query("MATCH (n) RETURN n LIMIT 10")

[{'n': {'id': 'fc837ab85ac4d9fe2135fc61e30f0d43',
   'text': 'Christopher Nolan directed Inception. Leonardo DiCaprio acted in Inception. Inception is a Science Fiction movie released in 2010.'}},
 {'n': {'id': 'Christopher Nolan'}},
 {'n': {'id': 'Leonardo Dicaprio'}},
 {'n': {'id': 'Inception'}},
 {'n': {'id': 'Science Fiction'}},
 {'n': {'id': '2e3fbf2f1831401c96c8d7e53e65e98b',
   'text': 'Christopher Nolan directed Interstellar. Matthew McConaughey acted in Interstellar. Interstellar is a Science Fiction movie released in 2014.'}},
 {'n': {'id': 'Matthew Mcconaughey'}},
 {'n': {'id': 'Interstellar'}},
 {'n': {'id': '89cb7e5b4a73bb0bf42e7e8233a48c4a',
   'text': 'Christopher Nolan directed The Dark Knight. Christian Bale acted in The Dark Knight. The Dark Knight is an Action movie released in 2008.'}},
 {'n': {'id': 'Christian Bale'}}]

In [15]:
graph.refresh_schema()
print(graph.schema)


Node properties:
Document {id: STRING, text: STRING}
Person {id: STRING}
Movie {id: STRING}
Genre {id: STRING}
Relationship properties:

The relationships:
(:Document)-[:MENTIONS]->(:Person)
(:Document)-[:MENTIONS]->(:Movie)
(:Document)-[:MENTIONS]->(:Genre)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)


In [ ]:
#Find every movie that a director directed. Then, for each movie, find the actors who acted in it. Finally, return the director, movie, and all actors together.
result = graph.query("""
MATCH (director:Person)-[:DIRECTED]->(movie:Movie)
OPTIONAL MATCH (actor:Person)-[:ACTED_IN]->(movie)
RETURN director.id AS director, movie.id AS movie, collect(actor.id) AS actors
""")

result

[{'director': 'Christopher Nolan',
  'movie': 'Inception',
  'actors': ['Leonardo Dicaprio']},
 {'director': 'Christopher Nolan',
  'movie': 'Interstellar',
  'actors': ['Matthew Mcconaughey']},
 {'director': 'Christopher Nolan',
  'movie': 'The Dark Knight',
  'actors': ['Christian Bale']}]

## Generation: let the LLM answer using the graph

GraphCypherQAChain does three simple things:

Reads the Neo4j schema
Converts the user question into Cypher
Executes Cypher and sends the retrieved facts to the LLM for the final answer

In [17]:
from langchain_neo4j import GraphCypherQAChain

chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True
)

In [18]:
response = chain.invoke({
    "query": "Which movies did Christopher Nolan direct?"
})

print(response["result"])

     



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {id: "Christopher Nolan"})-[:DIRECTED]->(m:Movie)
RETURN m
Full Context:
[{'m': {'id': 'Inception'}}, {'m': {'id': 'Interstellar'}}, {'m': {'id': 'The Dark Knight'}}]

> Finished chain.
Inception, Interstellar, The Dark Knight.


In [19]:
response = chain.invoke({
    "query": "Which movie did Christopher Nolan direct in 2010 and who acted in it?"
})

print(response["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {id: "Christopher Nolan"})-[:DIRECTED]->(m:Movie)
WHERE m.id CONTAINS "2010"
OPTIONAL MATCH (a:Person)-[:ACTED_IN]->(m)
RETURN m.id AS movieId, collect(a.id) AS actors;
Full Context:
[]

> Finished chain.
I don't know the answer.
